# Random Issue Sample by Decade

This notebook draws a reproducible random sample of Economist issues for each decade in a fixed year range. It uses the METS issue manifest as the source of available issue identifiers, parses issue dates from the `ECON-yyyy-mmdd` issue ids, and writes the sampled issues to `data/processed`.

## Parameters and Paths

Run this notebook from `code/scripts`. Change `N_ISSUES_PER_DECADE` to control the sample size. `END_YEAR` is inclusive, so the default range includes issues published in 2000.

In [ ]:
from pathlib import Path, PurePosixPath
import hashlib
import json

import pandas as pd

pd.options.display.max_columns = 80
pd.options.display.max_colwidth = 140

N_ISSUES_PER_DECADE = 5
RANDOM_SEED = 42
START_YEAR = 1940
END_YEAR = 2007

# Paths are relative to this notebook's directory: code/scripts.
issue_manifest_json = Path("../../data/metadata/economist_mets_issue_manifest.json")
sample_csv = Path(f"../../data/processed/random_issue_sample_by_decade_{START_YEAR}_{END_YEAR}.csv")

assert issue_manifest_json.exists(), f"Missing issue manifest: {issue_manifest_json}"
sample_csv.parent.mkdir(parents=True, exist_ok=True)

print(f"Issue manifest: {issue_manifest_json}")
print(f"Sample CSV:     {sample_csv}")

## Load and Validate the Issue Manifest

The manifest supplies the issue universe. The notebook treats the manifest order as source order but samples only from parsed issue ids, keeping the raw manifest unchanged.

In [ ]:
manifest = json.loads(issue_manifest_json.read_text(encoding="utf-8"))

required_manifest_keys = {"issue_count", "issue_ids", "year_issue_counts", "years"}
missing_manifest_keys = required_manifest_keys.difference(manifest)
assert not missing_manifest_keys, f"Missing manifest keys: {sorted(missing_manifest_keys)}"

issue_ids = pd.Series(manifest["issue_ids"], dtype="string", name="issue_id")
assert len(issue_ids) == manifest["issue_count"]
assert issue_ids.notna().all()
assert issue_ids.is_unique

issues = issue_ids.to_frame()
date_parts = issues["issue_id"].str.extract(r"^ECON-(\d{4})-(\d{4})$")
assert date_parts.notna().all().all(), "Every issue_id should match ECON-yyyy-mmdd."

issues["issue_date"] = pd.to_datetime(
    date_parts[0] + "-" + date_parts[1],
    format="%Y-%m%d",
    errors="coerce",
)
assert issues["issue_date"].notna().all(), "Every issue_id should contain a parseable issue date."

issues["year"] = issues["issue_date"].dt.year.astype(int)
issues["decade"] = (issues["year"] // 10 * 10).astype(int)
issues["metadata_path"] = [
    str(PurePosixPath("../../data/metadata") / str(year) / f"{issue_id}.mets.xml")
    for issue_id, year in zip(issues["issue_id"], issues["year"])
]

assert sorted(issues["year"].unique().tolist()) == manifest["years"]

manifest_year_counts = (
    pd.Series(manifest["year_issue_counts"], name="manifest_issues")
    .rename_axis("year")
    .reset_index()
)
manifest_year_counts["year"] = manifest_year_counts["year"].astype(int)
manifest_year_counts["manifest_issues"] = manifest_year_counts["manifest_issues"].astype(int)

observed_year_counts = (
    issues.groupby("year")
    .size()
    .rename("observed_issues")
    .reset_index()
)

year_count_check = observed_year_counts.merge(manifest_year_counts, on="year", how="outer")
assert (year_count_check["observed_issues"] == year_count_check["manifest_issues"]).all()

issues.head()

## Restrict to the Requested Year Range

The default range is 1900 through 2000 inclusive. The final decade is therefore a partial 2000s period containing only issues from 2000.

In [ ]:
assert START_YEAR <= END_YEAR
assert N_ISSUES_PER_DECADE > 0

eligible_issues = issues.loc[issues["year"].between(START_YEAR, END_YEAR)].copy()
assert len(eligible_issues) > 0
assert int(eligible_issues["year"].min()) == START_YEAR
assert int(eligible_issues["year"].max()) == END_YEAR

expected_decades = list(
    range((START_YEAR // 10) * 10, (END_YEAR // 10) * 10 + 1, 10)
)
observed_decades = sorted(eligible_issues["decade"].unique().tolist())
assert observed_decades == expected_decades, {
    "expected": expected_decades,
    "observed": observed_decades,
}

missing_metadata = eligible_issues.loc[
    ~eligible_issues["metadata_path"].map(lambda path: Path(path).exists()),
    ["issue_id", "metadata_path"],
]
assert missing_metadata.empty, missing_metadata.head(10).to_dict("records")

decade_overview = (
    eligible_issues.groupby("decade", as_index=False)
    .agg(
        available_issues=("issue_id", "size"),
        first_issue_date=("issue_date", "min"),
        last_issue_date=("issue_date", "max"),
    )
)
decade_overview["first_issue_date"] = decade_overview["first_issue_date"].dt.strftime("%Y-%m-%d")
decade_overview["last_issue_date"] = decade_overview["last_issue_date"].dt.strftime("%Y-%m-%d")

insufficient_decades = decade_overview.loc[
    decade_overview["available_issues"] < N_ISSUES_PER_DECADE
]
assert insufficient_decades.empty, insufficient_decades.to_dict("records")

decade_overview

## Draw the Decade Sample

A stable seeded random key is assigned to each issue from the pair `RANDOM_SEED` and `issue_id`. This makes the ordering independent of the current `START_YEAR` and `END_YEAR` filter. Within each decade, the notebook keeps the `N_ISSUES_PER_DECADE` lowest random keys and uses `issue_id` as a deterministic tie-breaker.

In [ ]:
def stable_random_key(issue_id: str, seed: int) -> int:
    digest = hashlib.blake2b(f"{seed}:{issue_id}".encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(digest, byteorder="big")


sample_pool = eligible_issues.assign(
    random_key=lambda df: df["issue_id"].map(lambda issue_id: stable_random_key(issue_id, RANDOM_SEED))
)

sampled_issues = (
    sample_pool.sort_values(["decade", "random_key", "issue_id"])
    .groupby("decade", group_keys=False)
    .head(N_ISSUES_PER_DECADE)
    .copy()
)
sampled_issues["sample_rank"] = sampled_issues.groupby("decade").cumcount() + 1
sampled_issues = sampled_issues.sort_values(["decade", "sample_rank"]).reset_index(drop=True)
sampled_issues["issue_date"] = sampled_issues["issue_date"].dt.strftime("%Y-%m-%d")
sampled_issues["sample_seed"] = RANDOM_SEED
sampled_issues["sample_n_per_decade"] = N_ISSUES_PER_DECADE

sampled_issues = sampled_issues.loc[
    :,
    [
        "decade",
        "sample_rank",
        "issue_id",
        "issue_date",
        "year",
        "metadata_path",
        "sample_seed",
        "sample_n_per_decade",
    ],
]

expected_sample_rows = N_ISSUES_PER_DECADE * len(expected_decades)
assert len(sampled_issues) == expected_sample_rows
assert (sampled_issues.groupby("decade").size() == N_ISSUES_PER_DECADE).all()
assert sampled_issues["issue_id"].is_unique

full_decades = [
    decade for decade in expected_decades
    if START_YEAR <= decade and END_YEAR >= decade + 9
]

reference_pool = issues.loc[issues["decade"].isin(full_decades)].assign(
    random_key=lambda df: df["issue_id"].map(lambda issue_id: stable_random_key(issue_id, RANDOM_SEED))
)
reference_full_decade_sample = (
    reference_pool.sort_values(["decade", "random_key", "issue_id"])
    .groupby("decade", group_keys=False)
    .head(N_ISSUES_PER_DECADE)
    .loc[:, ["decade", "issue_id"]]
    .reset_index(drop=True)
)
current_full_decade_sample = (
    sampled_issues.loc[sampled_issues["decade"].isin(full_decades), ["decade", "issue_id"]]
    .reset_index(drop=True)
)
pd.testing.assert_frame_equal(current_full_decade_sample, reference_full_decade_sample)

sampled_issues

## Write and Verify the Sample

The processed CSV contains one row per selected issue and enough sampling metadata to reproduce the draw.

In [ ]:
sampled_issues.to_csv(sample_csv, index=False)

reloaded_sample = pd.read_csv(
    sample_csv,
    dtype={"issue_id": "string", "metadata_path": "string"},
)
assert len(reloaded_sample) == len(sampled_issues)
assert list(reloaded_sample.columns) == list(sampled_issues.columns)
assert reloaded_sample["issue_id"].is_unique
assert (reloaded_sample.groupby("decade").size() == N_ISSUES_PER_DECADE).all()

print(f"Wrote {len(reloaded_sample):,} sampled issues to {sample_csv}")
reloaded_sample.head(10)

## Conclusion

The sampled issue list is ready for downstream workflows that need a balanced set of issues by decade. To sample only full twentieth-century decades, set `END_YEAR = 1999` before rerunning.